In [ ]:
import json

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

from analysis.src.data import geographic_features, radial_features
from analysis.src.io import experiment_paths, load_data
from analysis.src.constants import COMPONENT_LABELS, COMPONENTS, RADIAL_BAND_ORDER, RADIAL_BREAKPOINTS_KM, SECTOR_ORDER

In [ ]:
OUT = experiment_paths('geography/gradients_and_shortfalls')
SHORTFALL_THRESHOLD = 1.0

data = load_data()
walkability_sd = data.walkability_score.std(ddof = 1)

In [ ]:
# Model spatial gradients
spatial, centre = geographic_features(data, RADIAL_BREAKPOINTS_KM)
spatial['distance_km'] = spatial.dist_km
spatial['radial_band'] = pd.cut(spatial.dist_km, [-np.inf, *RADIAL_BREAKPOINTS_KM, np.inf], labels = RADIAL_BAND_ORDER)

radial_formula = 'walkability_score ~ seg1 + seg2 + seg3'
gradient_formula = 'walkability_score ~ (seg1 + seg2 + seg3) * C(sector)'

radial_model = smf.ols(radial_formula, data = spatial).fit()
gradient_model = smf.ols(gradient_formula, data = spatial).fit()

spatial['radial_expected'] = radial_model.predict(spatial)
residual = spatial.radial_expected - spatial.walkability_score
spatial['radial_deficit_z'] = (residual - residual.mean()) / residual.std(ddof = 0)

component_piecewise_rows = []
for component in COMPONENTS:
    model = smf.ols(f'{component} ~ seg1 + seg2 + seg3', data = spatial).fit()

    for band, term in zip(RADIAL_BAND_ORDER, ['seg1', 'seg2', 'seg3']):
        component_piecewise_rows.append(
            {
                'component': component,
                'radial_band': band,
                'slope_score_units_per_km': model.params[term],
                'slope_component_sd_per_km': model.params[term] / spatial[component].std(ddof = 1),
                'r_squared': model.rsquared,
                'n_lsoas': int(model.nobs)
            }
        )

    deficit = model.predict(spatial) - spatial[component]
    spatial[f'{component}_deficit_z'] = (deficit - deficit.mean()) / deficit.std(ddof = 0)

deficits = spatial[[f'{c}_deficit_z' for c in COMPONENTS]].rename(
    columns = {f'{c}_deficit_z': COMPONENT_LABELS[c] for c in COMPONENTS}
)

spatial['dominant_shortfall'] = deficits.idxmax(axis = 1).where(deficits.max(axis = 1) >= SHORTFALL_THRESHOLD, 'Mixed')
spatial['is_shortfall'] = spatial.radial_deficit_z >= SHORTFALL_THRESHOLD
shortfalls = spatial.loc[spatial.is_shortfall]

spatial.to_parquet(OUT.spatial_parquet)
spatial.dissolve(by = 'LAD22NM')[['geometry']].to_parquet(OUT.boroughs_parquet)

summary = {
    'centre': centre,
    'breakpoints_km': RADIAL_BREAKPOINTS_KM,
    'radial_r_squared': radial_model.rsquared,
    'sectoral_r_squared': gradient_model.rsquared,
    'n_lsoas': len(spatial),
    'n_shortfalls': len(shortfalls),
    'shortfall_fraction': len(shortfalls) / len(spatial),
    'gradient_formula': gradient_formula
}

(OUT.summary_json).write_text(json.dumps(summary, indent = 2) + '\n')

In [ ]:
# Estimate radial gradients and identify local shortfalls
curve = pd.DataFrame({'distance_km': np.linspace(0, spatial.dist_km.max(), 500)})

for column, values in zip(['seg1', 'seg2', 'seg3'], radial_features(curve.distance_km).T):
    curve[column] = values

curve['predicted'] = radial_model.predict(curve)
curve.to_csv(OUT.radial_curve_csv, index = False)

pd.DataFrame(
    {
        'band': RADIAL_BAND_ORDER,
        'start_km': [0, *RADIAL_BREAKPOINTS_KM],
        'end_km': [*RADIAL_BREAKPOINTS_KM, spatial.dist_km.max()],
        'slope_sd_per_km': radial_model.params[['seg1', 'seg2', 'seg3']].to_numpy() / walkability_sd
    }
).to_csv(OUT.radial_segments_csv, index = False)

spatial.groupby(['sector', 'radial_band'], observed = False).walkability_score.mean().unstack().reindex(
    index = SECTOR_ORDER, columns = RADIAL_BAND_ORDER
).to_csv(OUT.sector_band_csv, index_label = 'sector')

counts = shortfalls.dominant_shortfall.value_counts().rename_axis('component').reset_index(name = 'n_lsoas')
counts['fraction_of_shortfalls'] = counts.n_lsoas / len(shortfalls)
counts.to_csv(OUT.shortfall_summary_csv, index = False)

gradient_model.params.rename('coefficient').rename_axis('term').reset_index().to_csv(
    OUT.sectoral_coefficients_csv, index = False
)

display(summary)
display(counts)

In [ ]:
# Component gradients and directional comparisons
pd.DataFrame(component_piecewise_rows).to_csv(OUT.component_piecewise_gradients_csv, index = False)

radial_rows, sector_rows, sector_band_rows, sector_mean_rows = [], [], [], []
for component in COMPONENTS:
    model = smf.ols(f'{component} ~ dist_km', data = spatial).fit()

    radial_rows.append(
        {
            'component': component,
            'slope_score_units_per_km': model.params['dist_km'],
            'r_squared': model.rsquared,
            'n_lsoas': int(model.nobs)
        }
    )

    for sector, sample in spatial.groupby('sector', observed = True):
        model = smf.ols(f'{component} ~ dist_km', data = sample).fit()

        sector_rows.append(
            {
                'component': component,
                'sector': sector,
                'slope_score_units_per_km': model.params['dist_km'],
                'r_squared': model.rsquared,
                'n_lsoas': int(model.nobs)
            }
        )

    for (sector, band), sample in spatial.groupby(['sector', 'radial_band'], observed = True):
        model = smf.ols(f'{component} ~ dist_km', data = sample).fit()

        sector_band_rows.append(
            {
                'component': component,
                'sector': sector,
                'radial_band': band,
                'slope_score_units_per_km': model.params['dist_km'],
                'r_squared': model.rsquared,
                'n_lsoas': int(model.nobs)
            }
        )

        sector_mean_rows.append(
            {
                'component': component,
                'sector': sector,
                'radial_band': band,
                'mean_score': sample[component].mean(),
                'n_lsoas': int(sample[component].notna().sum())
            }
        )

for name, rows in [
    ('component_radial_gradients', radial_rows),
    ('component_sector_gradients', sector_rows),
    ('component_sector_band_gradients', sector_band_rows),
    ('component_sector_band_means', sector_mean_rows)
]:
    pd.DataFrame(rows).to_csv(OUT.tables / f'{name}.csv', index = False)

display(pd.DataFrame(radial_rows))